# 02 · 数据清洗与合并（Clean & Merge）

**目标**：把 **FBref + Understat + Transfermarkt** 三个来源合并成**一张干净的总表**，每名球员一行。

| 步骤 | 内容 | 对应的课 |
|---|---|---|
| 1 | 读取原始数据 | 0-A pandas |
| 2 | FBref：两张表合并 + 转会球员加总 | `merge`、`groupby` |
| 3 | Understat：转会球员加总 | `groupby` |
| 4 | 球会别名表 | 名字配对第 3 层 |
| 5 | 配对 FBref ↔ Understat（名字 + 球会） | 漏斗配对 |
| 6 | 配对 FBref ↔ Transfermarkt（名字 + 出生年份） | 漏斗配对 |
| 7 | 取得赛季结束时的身价 | 时间序列处理 |
| 8 | 合并成总表、分位置 | `merge` |
| 9 | 过滤 + per-90 换算 | Striker F 的教训 |
| 10 | 保存 + 输出需要人工检查的清单 | 数据质量 |

⚠️ `data/raw/` 里的文件**只读不改**，所有结果都存进 `data/processed/`。

In [26]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from unidecode import unidecode
from rapidfuzz import process, fuzz

pd.set_option("display.max_columns", 50)

# 自动找到项目根目录（有 requirements.txt 的那个文件夹）
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
TM_DIR = RAW / "transfermarket"          # 你的文件夹名（多一个 e）
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

SEASON_END = pd.Timestamp("2026-06-30")  # 2025-26 赛季结束日
MIN_MINUTES = 900                        # 最低出场时间门槛
print("项目根目录:", ROOT)

项目根目录: c:\Users\hengk\OneDrive\Desktop\Football_Scount


## 1️⃣ 读取原始数据

FBref 的 CSV 第 1 行是分组标题（Performance、Playing Time…），第 2 行才是真正的列名，所以用 `header=1`。
最后一列 `-9999` 其实是 **FBref 球员 ID**，改名为 `fbref_id`。

In [27]:
fb_std  = pd.read_csv(RAW / "fbref_standard_2025_26.csv", header=1).rename(columns={"-9999": "fbref_id"})
fb_misc = pd.read_csv(RAW / "fbref_misc_2025_26.csv",     header=1).rename(columns={"-9999": "fbref_id"})
us_raw  = pd.read_csv(RAW / "understat_2025_26.csv")
tm_players = pd.read_csv(TM_DIR / "players.csv")
tm_vals    = pd.read_csv(TM_DIR / "player_valuations.csv", parse_dates=["date"])

for name, df in [("FBref standard", fb_std), ("FBref misc", fb_misc), ("Understat", us_raw),
                 ("TM players", tm_players), ("TM valuations", tm_vals)]:
    print(f"{name:15} {df.shape}")

FBref standard  (2839, 27)
FBref misc      (2839, 23)
Understat       (2775, 23)
TM players      (50149, 26)
TM valuations   (656301, 6)


## 2️⃣ FBref：合并两张表 + 转会球员加总

两张表的行顺序完全一样（同一个球员、同一个球会），所以用 `fbref_id + Squad + Comp` 合并。

🍜 **转会球员**：Himad Abdelli 在马赛和昂热各有一行，就像一个员工在两家分店打工，计算全年工时要**加总**。
- **次数类**（分钟、进球、拦截…）→ `sum` 加总
- **主要球会 / 联赛** → 取**上场分钟最多**的那一个

In [28]:
misc_cols = ["fbref_id", "Squad", "Comp", "Fls", "Fld", "Off", "Crs", "Int", "TklW"]
fb = fb_std.merge(fb_misc[misc_cols], on=["fbref_id", "Squad", "Comp"], how="left", validate="one_to_one")

LEAGUE_MAP = {
    "eng Premier League": "ENG-Premier League", "es La Liga": "ESP-La Liga",
    "de Bundesliga": "GER-Bundesliga", "it Serie A": "ITA-Serie A", "fr Ligue 1": "FRA-Ligue 1",
}
fb["league"] = fb["Comp"].map(LEAGUE_MAP)

FB_SUM = ["MP", "Starts", "Min", "Gls", "Ast", "PK", "PKatt", "CrdY", "CrdR",
          "Fls", "Fld", "Off", "Crs", "Int", "TklW"]
fb[FB_SUM] = fb[FB_SUM].apply(pd.to_numeric, errors="coerce").fillna(0)

# 主要球会 = 分钟最多的那一行
main = fb.sort_values("Min", ascending=False).drop_duplicates("fbref_id")[
    ["fbref_id", "Player", "Nation", "Pos", "Born", "Squad", "league"]]
totals = fb.groupby("fbref_id")[FB_SUM].sum().reset_index()
n_clubs = fb.groupby("fbref_id")["Squad"].nunique().rename("n_clubs").reset_index()

fb_players = main.merge(totals, on="fbref_id").merge(n_clubs, on="fbref_id")
print("FBref：", len(fb), "行 →", len(fb_players), "名球员；其中转会球员", (fb_players.n_clubs > 1).sum(), "人")
display(fb_players[fb_players.n_clubs > 1].head(3))

FBref： 2839 行 → 2691 名球员；其中转会球员 148 人


,fbref_id,Player,Nation,Pos,Born,Squad,league,MP,Starts,Min,Gls,Ast,PK,PKatt,CrdY,CrdR,Fls,Fld,Off,Crs,Int,TklW,n_clubs
295,c4f5eb61,Urko González,es ESP,MF,2001.0,Espanyol,ESP-La Liga,36,28,2634,0,1,0,0,7,0,42,42,1,12,29,33,2
340,8794e251,Adrien Rabiot,fr FRA,MF,1995.0,Milan,ITA-Serie A,30,29,2640,6,4,0,0,7,1,37,56,7,67,20,31,2
366,85ce1949,Esteban Lepaul,fr FRA,FW,2000.0,Rennes,FRA-Ligue 1,34,33,2685,21,5,3,4,3,0,25,16,9,17,3,5,2


## 3️⃣ Understat：转会球员加总

和 FBref 同样的逻辑。Understat 也有自己的 `player_id`。

In [29]:
US_SUM = ["matches", "minutes", "goals", "xg", "np_goals", "np_xg", "assists", "xa",
          "shots", "key_passes", "yellow_cards", "red_cards", "xg_chain", "xg_buildup"]
us_main = us_raw.sort_values("minutes", ascending=False).drop_duplicates("player_id")[
    ["player_id", "player", "team", "league", "position"]]
us_players = us_main.merge(us_raw.groupby("player_id")[US_SUM].sum().reset_index(), on="player_id")
us_players = us_players.rename(columns={"player_id": "us_id", "player": "us_name", "team": "us_team",
                                        "league": "us_league",
                                        "position": "us_position", "minutes": "us_minutes"})
print("Understat：", len(us_raw), "行 →", len(us_players), "名球员")

Understat： 2775 行 → 2693 名球员


## 4️⃣ 名字标准化 + 球会别名表

和第 0 课一样：去重音、小写、去符号、**单词排序**。

96 支球队里有 21 支两个网站叫法不同，例如 `Manchester United` ↔ `Manchester Utd`。
下面的别名表把 **Understat 的球会名 → FBref 的球会名**。最后用 `assert` 检查**每一支都对上了**，漏了就会报错。

In [30]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    name = unidecode(str(name)).lower()
    name = re.sub(r"[^a-z\s]", " ", name)
    return " ".join(sorted(name.split()))

US_TO_FB_TEAM = {
    "AC Milan": "Milan", "Alaves": "Alavés", "Atletico Madrid": "Atlético Madrid",
    "Bayer Leverkusen": "Leverkusen", "Borussia Dortmund": "Dortmund",
    "Borussia M.Gladbach": "Gladbach", "Eintracht Frankfurt": "Frankfurt",
    "FC Cologne": "Köln", "FC Heidenheim": "Heidenheim", "Leeds": "Leeds United",
    "Manchester United": "Manchester Utd", "Newcastle United": "Newcastle",
    "Nottingham Forest": "Nottingham", "Paris Saint Germain": "Paris SG",
    "Parma Calcio 1913": "Parma", "RasenBallsport Leipzig": "RB Leipzig",
    "Real Oviedo": "Oviedo", "St. Pauli": "St Pauli", "Verona": "Hellas Verona",
    "VfB Stuttgart": "Stuttgart", "Wolverhampton Wanderers": "Wolves",
}
us_raw["team_fb"] = us_raw["team"].replace(US_TO_FB_TEAM)

missing = set(us_raw["team_fb"]) - set(fb["Squad"])
assert not missing, f"还有球会没对上: {missing}"
print("✅ 96 支球队全部对上")

✅ 96 支球队全部对上


## 5️⃣ 配对 FBref ↔ Understat

用**每一行（球员 × 球会）**来配对，因为球会能帮忙排除同名的人（还记得两个 Danilo 吗？）。

| 层级 | 方法 |
|---|---|
| 1️⃣ exact | 同联赛 + 同球会 + 标准化名字完全相同 |
| 2️⃣ fuzzy | 同球会内，名字相似度 ≥ 85，**而且上场分钟差距 ≤ 10%（至少容许 90 分钟）** |

🔍 **为什么要比对分钟？** 测试时发现 `Oliver Jensen` ↔ `Oliver Sørensen` 只有 85.7 分，看起来很可疑。
但两人都在 Parma，FBref 记 1840 分钟、Understat 记 1834 分钟，**几乎一样**，所以其实是同一个人。
两个独立来源的分钟数对得上，就是很强的证据。这叫 **cross-validation between sources（跨来源验证）**。

`token_set_ratio` 的特点：只要**一边的单词全部出现在另一边**就给高分，所以
`Kylian Mbappé` ↔ `Kylian Mbappe-Lottin` 会得 100 分。

In [31]:
fb["name_key"] = fb["Player"].map(normalize_name)
us_raw["name_key"] = us_raw["player"].map(normalize_name)

# 第 1 层：完全相同
l1 = fb[["fbref_id", "league", "Squad", "name_key"]].merge(
    us_raw[["player_id", "league", "team_fb", "name_key"]],
    left_on=["league", "Squad", "name_key"], right_on=["league", "team_fb", "name_key"])
l1 = l1[["fbref_id", "player_id"]].assign(method="exact", score=100.0)

# 第 2 层：同球会内模糊匹配
fb_left = fb[~fb["fbref_id"].isin(l1["fbref_id"])]
us_left = us_raw[~us_raw["player_id"].isin(l1["player_id"])]
rows = []
for _, r in fb_left.iterrows():
    cands = us_left[(us_left["team_fb"] == r["Squad"]) & (us_left["league"] == r["league"])]
    if cands.empty:
        continue
    best = process.extractOne(r["name_key"], cands["name_key"], scorer=fuzz.token_set_ratio)
    if not best or best[1] < 85:
        continue
    us_min = cands.loc[best[2], "minutes"]
    if abs(us_min - r["Min"]) <= max(90, 0.10 * r["Min"]):     # 分钟数验证
        rows.append({"fbref_id": r["fbref_id"], "player_id": cands.loc[best[2], "player_id"],
                     "method": "fuzzy", "score": best[1]})
    else:
        print(f"❌ 拒绝：{r['Player']} ↔ {cands.loc[best[2], 'player']}（分钟 {r['Min']:.0f} vs {us_min}）")
l2 = pd.DataFrame(rows, columns=["fbref_id", "player_id", "method", "score"])

links_us = pd.concat([l1, l2]).sort_values("score", ascending=False)
# 一对一：同一个 ID 只能配一个人（转会球员会出现两次相同的配对，去重即可）
links_us = links_us.drop_duplicates(["fbref_id", "player_id"])
conflict = links_us["fbref_id"].duplicated(keep=False) | links_us["player_id"].duplicated(keep=False)
print("⚠️ 一对多冲突：", conflict.sum(), "行（保留分数最高的）")
links_us = links_us.drop_duplicates("fbref_id").drop_duplicates("player_id")
links_us = links_us.rename(columns={"player_id": "us_id", "method": "us_method", "score": "us_score"})

print(f"✅ exact {(links_us.us_method=='exact').sum()} 人  🔍 fuzzy {(links_us.us_method=='fuzzy').sum()} 人")
print(f"配对率：{len(links_us)}/{len(fb_players)} = {len(links_us)/len(fb_players):.1%}")

# 抽查模糊配对的结果
check = links_us[links_us.us_method == "fuzzy"].merge(fb_players[["fbref_id", "Player"]]) \
                                               .merge(us_players[["us_id", "us_name"]])
display(check[["Player", "us_name", "us_score"]].sort_values("us_score").head(10))

❌ 拒绝：Eren Dinkçi ↔ Eren Sami Dinkci（分钟 1131 vs 1401）
❌ 拒绝：Karl Etta Eyong ↔ Etta Eyong（分钟 1416 vs 1657）
❌ 拒绝：João Mário Lopes ↔ João Mário（分钟 735 vs 1003）
❌ 拒绝：Raúl ↔ Raul García（分钟 1041 vs 926）
❌ 拒绝：Ibrahim Sulemana ↔ Sulemana（分钟 45 vs 866）
❌ 拒绝：Ali Yousuf ↔ Ali Youssef（分钟 969 vs 90）
⚠️ 一对多冲突： 0 行（保留分数最高的）
✅ exact 2435 人  🔍 fuzzy 185 人
配对率：2620/2691 = 97.4%


,Player,us_name,us_score
182,Joe Gomez,Joseph Gomez,85.714286
183,Tino Livramento,Valentino Livramento,85.714286
184,Oliver Jensen,Oliver Sørensen,85.714286
181,Andy Robertson,Andrew Robertson,86.666667
180,Nayef Aguerd,Naif Aguerd,86.956522
179,Tay Abed,Tai Abed,87.500000
178,Daniel Ballard,Dan Ballard,88.000000
175,Toni Martínez,Antonio Martínez,89.655172
176,Santi Cazorla,Santiago Cazorla,89.655172
173,Dapo Afolayan,Oladapo Afolayan,89.655172


## 6️⃣ 配对 FBref ↔ Transfermarkt

Transfermarkt 的“当前球会”是 **2026 年 7 月**的，夏天转会的球员球会会对不上，所以这里改用**出生年份**做辅助验证。

| 层级 | 方法 |
|---|---|
| 1️⃣ exact | 标准化名字 + 出生年份完全相同，而且**只有一个**候选人 |
| 2️⃣ fuzzy | 同出生年份内，相似度 ≥ 90，而且**第一名明显领先第二名** |

候选人范围：`last_season >= 2024` 的所有球员（包括夏天离开五大联赛的人）。

🛡️ **为什么要“只有一个候选人”？** 巴西球员常用单名（Danilo、Alysson…），同年出生又同名的人确实存在。
宁可**不配**，也不要**配错**，配错的数据会悄悄污染模型。

In [32]:
tm = tm_players[tm_players["last_season"] >= 2024].copy()
tm["name_key"] = tm["name"].map(normalize_name)
tm["birth_year"] = pd.to_datetime(tm["date_of_birth"]).dt.year
fb_players["name_key"] = fb_players["Player"].map(normalize_name)

# 第 1 层：名字 + 出生年份，而且候选人唯一
cand = fb_players[["fbref_id", "name_key", "Born"]].merge(
    tm[["player_id", "name_key", "birth_year"]],
    left_on=["name_key", "Born"], right_on=["name_key", "birth_year"])
n_cand = cand.groupby("fbref_id")["player_id"].transform("count")
t1 = cand[n_cand == 1][["fbref_id", "player_id"]].assign(method="exact", score=100.0)
ambiguous_tm = cand[n_cand > 1]
print("同名同年、无法自动判断：", ambiguous_tm["fbref_id"].nunique(), "人")

# 第 2 层：同出生年份内模糊匹配
done = set(t1["fbref_id"]) | set(ambiguous_tm["fbref_id"])
rows = []
for _, r in fb_players[~fb_players["fbref_id"].isin(done)].iterrows():
    pool = tm[(tm["birth_year"] == r["Born"]) & (~tm["player_id"].isin(t1["player_id"]))]
    if pool.empty:
        continue
    top = process.extract(r["name_key"], pool["name_key"], scorer=fuzz.token_set_ratio, limit=2)
    best, second = top[0], (top[1] if len(top) > 1 else (None, 0, None))
    if best[1] >= 90 and best[1] - second[1] >= 5:          # 明显领先第二名
        rows.append({"fbref_id": r["fbref_id"], "player_id": pool.loc[best[2], "player_id"],
                     "method": "fuzzy", "score": best[1]})
t2 = pd.DataFrame(rows, columns=["fbref_id", "player_id", "method", "score"])

links_tm = pd.concat([t1, t2]).sort_values("score", ascending=False)
links_tm = links_tm.drop_duplicates("fbref_id").drop_duplicates("player_id")
links_tm = links_tm.rename(columns={"player_id": "tm_id", "method": "tm_method", "score": "tm_score"})

print(f"✅ exact {(links_tm.tm_method=='exact').sum()} 人  🔍 fuzzy {(links_tm.tm_method=='fuzzy').sum()} 人")
print(f"配对率：{len(links_tm)}/{len(fb_players)} = {len(links_tm)/len(fb_players):.1%}")

同名同年、无法自动判断： 5 人
✅ exact 2495 人  🔍 fuzzy 111 人
配对率：2606/2691 = 96.8%


## 7️⃣ 赛季结束时的身价（Target）

`player_valuations.csv` 记录了每名球员**每一次**身价更新。我们要的是：

| 列 | 意思 |
|---|---|
| `value_end` | **2026 年 1 月 1 日 ~ 6 月 30 日之间最后一次**身价 → 模型要预测的 **Target** |
| `value_start` | **2025 年 8 月 1 日之前最后一次**身价 → 赛季开始时的身价（之后可以算涨跌） |

🍜 比喻：进货价（赛季开始）和年底盘点价（赛季结束）是两个不同时间点的价格，不能混用。
身价以**百万欧元（M€）**为单位，比较好读。

In [33]:
v = tm_vals[tm_vals["player_id"].isin(links_tm["tm_id"])].sort_values("date")

end = v[(v["date"] >= "2026-01-01") & (v["date"] <= SEASON_END)] \
        .groupby("player_id").last()[["date", "market_value_in_eur"]] \
        .rename(columns={"date": "value_end_date", "market_value_in_eur": "value_end"})
start = v[v["date"] < "2025-08-01"].groupby("player_id").last()[["market_value_in_eur"]] \
        .rename(columns={"market_value_in_eur": "value_start"})

values = end.join(start, how="outer")
values[["value_end", "value_start"]] = values[["value_end", "value_start"]] / 1e6
values = values.reset_index().rename(columns={"player_id": "tm_id"})
print("有赛季结束身价的球员：", values["value_end"].notna().sum())
print(values["value_end_date"].dt.to_period("M").value_counts().sort_index().tail(4))

有赛季结束身价的球员： 2398
value_end_date
2026-03       4
2026-04       2
2026-05     978
2026-06    1411
Freq: M, Name: count, dtype: int64


## 7️⃣b 射门风格指标（来自 Understat 每一脚射门）

阶段 4 发现 Barcola（边锋）被推荐成 Haaland 的平替，因为只看“射了多少”，不知道“**在哪里射、怎么射**”。
这一步从每一脚射门算出**风格指标**：

| 指标 | 意思 | 分辨什么 |
|---|---|---|
| `header_share` | 头球占射门的比例 | 中锋（空中能力） vs 边锋 |
| `avg_shot_dist` | 平均射门距离（米） | 禁区内射手 vs 远射型 |
| `xg_per_shot` | 每脚射门的平均 xG | 射门**质量** |
| `unassisted_share` | 运动战中**无人助攻**的射门比例 | 自己创造机会 / 盘带的**近似指标** |

⚠️ **两个数据陷阱**（检查数据时发现的，报告可以写）：
- `body_part` 只有左脚 / 右脚，**空值其实是头球**：这些射门离球门更近，而且多数来自角球
- `situation` 的空值全部在**点球点**、xG ≈ 0.76，所以是**点球** → 排除，和 npxG 的逻辑一致

射门少于 10 脚的球员，比例不可靠（Striker F 的教训），设为空值，之后用同位置中位数补。

In [34]:
SHOTS_FILE = RAW / "understat_shots_2025_26.csv"
SHOT_COLS = ["header_share", "avg_shot_dist", "xg_per_shot", "unassisted_share"]

if SHOTS_FILE.exists():
    sh = pd.read_csv(SHOTS_FILE)
    print("各联赛射门数：", sh["league"].value_counts().to_dict())
    sh = sh[sh["situation"].notna()].copy()                 # 排除点球
    sh["is_header"] = sh["body_part"].isna()                 # 空值 = 头球
    sh["dist"] = np.sqrt(((1 - sh["location_x"]) * 105) ** 2 + ((sh["location_y"] - 0.5) * 68) ** 2)
    open_play = sh["situation"] == "Open Play"
    sh["unassisted"] = sh["assist_player_id"].isna()

    g = sh.groupby("player_id")
    shot_feats = pd.DataFrame({
        "np_shots": g.size(),
        "header_share": g["is_header"].mean(),
        "avg_shot_dist": g["dist"].mean(),
        "xg_per_shot": g["xg"].mean(),
        "unassisted_share": sh[open_play].groupby("player_id")["unassisted"].mean(),
    })
    shot_feats.loc[shot_feats["np_shots"] < 10, SHOT_COLS] = np.nan
    shot_feats = shot_feats.round(3).reset_index().rename(columns={"player_id": "us_id"})
    print("有射门风格数据的球员（≥10 脚）：", shot_feats["header_share"].notna().sum())
else:
    print("⚠️ 找不到射门数据，先运行 01b_collect_understat_shots.ipynb；这次先跳过")
    shot_feats = pd.DataFrame(columns=["us_id", "np_shots"] + SHOT_COLS)

各联赛射门数： {'ENG-Premier League': 9524, 'ESP-La Liga': 9523, 'ITA-Serie A': 9373, 'GER-Bundesliga': 8124, 'FRA-Ligue 1': 7617}
有射门风格数据的球员（≥10 脚）： 1391


## 8️⃣ 合并成总表 + 分位置

以 **FBref 为主表**（`left` merge），再接上 Understat 和 Transfermarkt。

**位置分组**用 Transfermarkt 的细分位置（我们之前决定的）：

| `pos_group` | 细分位置 |
|---|---|
| `pos_group` | `role`（更细的角色） | 细分位置 |
|---|---|---|
| FW 前锋 | **ST** 中锋 | Centre-Forward、Second Striker |
| FW 前锋 | **W** 边锋 | Left / Right Winger |
| MF 中场 | **AM** 前腰 | Attacking Midfield |
| MF 中场 | **CM** 中前卫 / 后腰 | Central / Defensive Midfield、Left / Right Midfield |
| DF 后卫 | **CB** 中卫 | Centre-Back |
| DF 后卫 | **FB** 边后卫 | Left-Back、Right-Back |

`role` 是阶段 4 你提出的改进：**中锋只和中锋比、边锋只和边锋比**（App 里可以关掉，允许跨位置）。

`is_centre_back = True` 的球员，之后会用**身高**当作争顶能力的**近似指标**（报告里要注明只是近似）。

In [35]:
tm_cols = ["player_id", "name", "date_of_birth", "position", "sub_position", "foot",
           "height_in_cm", "current_club_name", "contract_expiration_date"]
tm_info = tm_players[tm_cols].rename(columns={"player_id": "tm_id", "name": "tm_name",
                                              "position": "tm_position", "current_club_name": "tm_club_jul26"})

df = (fb_players
      .merge(links_us, on="fbref_id", how="left")
      .merge(us_players, on="us_id", how="left")
      .merge(links_tm, on="fbref_id", how="left")
      .merge(tm_info, on="tm_id", how="left")
      .merge(values, on="tm_id", how="left")
      .merge(shot_feats, on="us_id", how="left"))

POS_GROUP = {
    "Centre-Forward": "FW", "Second Striker": "FW", "Left Winger": "FW", "Right Winger": "FW",
    "Attacking Midfield": "MF", "Central Midfield": "MF", "Defensive Midfield": "MF",
    "Left Midfield": "MF", "Right Midfield": "MF",
    "Centre-Back": "DF", "Left-Back": "DF", "Right-Back": "DF",
    "Goalkeeper": "GK",
}
df["pos_group"] = df["sub_position"].map(POS_GROUP)
ROLE = {
    "Centre-Forward": "ST", "Second Striker": "ST", "Left Winger": "W", "Right Winger": "W",
    "Attacking Midfield": "AM", "Central Midfield": "CM", "Defensive Midfield": "CM",
    "Left Midfield": "CM", "Right Midfield": "CM",
    "Centre-Back": "CB", "Left-Back": "FB", "Right-Back": "FB", "Goalkeeper": "GK",
}
df["role"] = df["sub_position"].map(ROLE)
df["is_centre_back"] = df["sub_position"].eq("Centre-Back")

# 年龄：以赛季结束日计算（比 FBref 的整数年龄更精确）
dob = pd.to_datetime(df["date_of_birth"])
df["age"] = ((SEASON_END - dob).dt.days / 365.25).round(1)

assert df["fbref_id"].is_unique, "每名球员应该只有一行！"
print("总表：", df.shape)

总表： (2691, 68)


## 9️⃣ 过滤 + per-90 换算

| 过滤条件 | 原因 |
|---|---|
| 排除门将 | 门将的数据维度完全不同 |
| FBref 出场 ≥ 900 分钟 | Striker F 的教训：样本太小，per-90 数据不可靠 |
| 三个来源都要配对成功 | 缺任何一边，就缺了一部分指标 |
| 要有 `value_end` 和 `pos_group` | 身价模型需要 Target，相似度需要位置 |

每一步都打印剩下多少人，这在报告里叫 **data funnel（数据漏斗）**，评审很喜欢看。

In [36]:
funnel = [("FBref 全部球员", len(df))]
is_gk = df["pos_group"].eq("GK") | (df["pos_group"].isna() & df["Pos"].eq("GK"))
d = df[~is_gk];                              funnel.append(("排除门将", len(d)))
d = d[d["Min"] >= MIN_MINUTES];              funnel.append((f"出场 ≥ {MIN_MINUTES} 分钟", len(d)))
d = d[d["us_id"].notna()];                   funnel.append(("Understat 配对成功", len(d)))
d = d[d["tm_id"].notna()];                   funnel.append(("Transfermarkt 配对成功", len(d)))
d = d[d["value_end"].notna() & d["pos_group"].notna()]
funnel.append(("有赛季结束身价 + 位置", len(d)))

display(pd.DataFrame(funnel, columns=["步骤", "剩下的球员"]))
players = d.copy()

,步骤,剩下的球员
0,FBref 全部球员,2691
1,排除门将,2497
2,出场 ≥ 900 分钟,1484
3,Understat 配对成功,1441
4,Transfermarkt 配对成功,1412
5,有赛季结束身价 + 位置,1410


In [37]:
# per-90：次数 ÷ 分钟 × 90（分钟统一用 FBref 的）
P90_COLS = {
    # Understat
    "np_xg": "npxg", "xa": "xa", "shots": "shots", "key_passes": "key_passes",
    "xg_chain": "xg_chain", "xg_buildup": "xg_buildup",
    # FBref
    "Int": "int", "TklW": "tklw", "Crs": "crs", "Fls": "fls", "Fld": "fld", "Off": "off",
    "Gls": "goals", "Ast": "assists",
}
for src, name in P90_COLS.items():
    players[f"{name}_p90"] = (players[src] / players["Min"] * 90).round(3)

display(players.groupby("pos_group")[["npxg_p90", "xa_p90", "int_p90", "tklw_p90"]].mean().round(2))

,npxg_p90,xa_p90,int_p90,tklw_p90
pos_group,,,,
DF,0.06,0.07,1.05,1.07
FW,0.32,0.16,0.37,0.62
MF,0.11,0.12,0.88,1.10


🤔 **合理性检查**：看上面这张表，前锋的 npxG 应该最高、后卫的拦截应该最多。如果不是，代表前面某一步出错了。

## 🔟 保存

| 文件 | 内容 |
|---|---|
| `players_2025_26.csv` | ✅ **最终总表**，阶段 3 起都用它 |
| `review_unmatched.csv` | ≥ 900 分钟但**没配对上**的球员，可以人工检查 |
| `review_fuzzy.csv` | 靠**模糊匹配**配对的球员，建议抽查 |

In [38]:
KEEP = (["fbref_id", "us_id", "tm_id", "Player", "Nation", "league", "Squad", "n_clubs",
         "pos_group", "role", "sub_position", "is_centre_back", "foot", "height_in_cm", "age",
         "Min", "MP", "Starts", "value_end", "value_end_date", "value_start",
         "contract_expiration_date", "tm_club_jul26"]
        + [f"{n}_p90" for n in P90_COLS.values()]
        + ["np_shots"] + SHOT_COLS
        + ["us_method", "us_score", "tm_method", "tm_score"])
final = players[KEEP].rename(columns={"Player": "player", "Nation": "nation", "Squad": "squad",
                                      "Min": "minutes", "MP": "matches", "Starts": "starts"})
final.to_csv(OUT / "players_2025_26.csv", index=False, encoding="utf-8")

base = df[~is_gk & (df["Min"] >= MIN_MINUTES)]
unmatched = base[base["us_id"].isna() | base["tm_id"].isna() | base["value_end"].isna()]
unmatched[["fbref_id", "Player", "Born", "Squad", "league", "Min", "us_id", "tm_id", "value_end"]] \
    .to_csv(OUT / "review_unmatched.csv", index=False, encoding="utf-8")

fuzzy = final[(final["us_method"] == "fuzzy") | (final["tm_method"] == "fuzzy")]
fuzzy.merge(df[["fbref_id", "us_name", "tm_name"]], on="fbref_id")[
    ["player", "us_name", "us_score", "tm_name", "tm_score", "squad"]] \
    .to_csv(OUT / "review_fuzzy.csv", index=False, encoding="utf-8")

print("✅ 最终总表：", final.shape)
print("📝 待检查：未配对", len(unmatched), "人；模糊配对", len(fuzzy), "人")
print(final["role"].value_counts())
display(final.head())

✅ 最终总表： (1410, 46)
📝 待检查：未配对 74 人；模糊配对 102 人
role
CM    368
CB    311
FB    254
W     195
ST    192
AM     90
Name: count, dtype: int64


,fbref_id,us_id,tm_id,player,nation,league,squad,n_clubs,pos_group,role,sub_position,is_centre_back,foot,height_in_cm,age,minutes,matches,starts,value_end,value_end_date,value_start,contract_expiration_date,tm_club_jul26,npxg_p90,xa_p90,shots_p90,key_passes_p90,xg_chain_p90,xg_buildup_p90,int_p90,tklw_p90,crs_p90,fls_p90,fld_p90,off_p90,goals_p90,assists_p90,np_shots,header_share,avg_shot_dist,xg_per_shot,unassisted_share,us_method,us_score,tm_method,tm_score
9,e06683ca,833.0,139208.0,Virgil van Dijk,nl NED,ENG-Premier League,Liverpool,1,DF,CB,Centre-Back,True,right,195.0,35.0,3420,38,38,15.0,2026-06-03,23.0,2027-06-30 00:00:00,Liverpool FC,0.129,0.009,0.789,0.289,0.329,0.300,0.684,0.395,0.026,0.500,0.474,0.026,0.158,0.000,30.0,0.80,10.316,0.163,0.250,exact,100.0,exact,100.0
12,4e015693,7438.0,505219.0,James Garner,eng ENG,ENG-Premier League,Everton,1,MF,CM,Defensive Midfield,False,right,182.0,25.3,3413,38,38,45.0,2026-06-03,18.0,2030-06-30 00:00:00,Everton FC,0.055,0.160,1.187,1.477,0.298,0.220,1.503,1.899,5.406,1.028,1.108,0.000,0.053,0.185,45.0,0.00,23.789,0.046,0.348,exact,100.0,exact,100.0
13,79c84d1c,1776.0,314875.0,Jarrod Bowen,eng ENG,ENG-Premier League,West Ham,1,FW,W,Right Winger,False,left,176.0,29.5,3403,38,38,30.0,2026-06-03,40.0,2030-06-30 00:00:00,West Ham United,0.199,0.223,2.089,1.164,0.380,0.103,0.714,0.741,3.914,0.502,1.296,0.450,0.238,0.291,78.0,0.09,17.503,0.096,0.162,exact,100.0,exact,100.0
14,5d0e0a1f,8640.0,654539.0,Adrien Truffert,fr FRA,ENG-Premier League,Bournemouth,1,DF,FB,Left-Back,False,left,176.0,24.6,3379,38,38,30.0,2026-06-03,15.0,2030-06-30 00:00:00,AFC Bournemouth,0.053,0.117,0.426,0.879,0.406,0.265,1.145,1.358,3.782,1.039,0.852,0.000,0.027,0.133,16.0,0.00,21.275,0.124,0.500,exact,100.0,exact,100.0
15,bee704fc,6174.0,413507.0,Nikola Milenković,rs SRB,ENG-Premier League,Nottingham,1,DF,CB,Centre-Back,True,right,195.0,28.7,3375,38,37,28.0,2026-06-03,35.0,2029-06-30 00:00:00,Nottingham Forest,0.014,0.007,0.400,0.080,0.215,0.213,0.773,0.507,0.053,0.560,0.533,0.027,0.000,0.000,16.0,0.75,20.080,0.032,0.000,exact,100.0,exact,100.0
